# 36. The Berth & Quay Length Design Problem

## Tier 1: The Pen & Paper Method (Robust Optimization Formulation)

### Goal
Formulate and solve the berth and quay length design problem using robust optimization to ensure optimal infrastructure design while protecting against uncertainty in vessel arrivals and service patterns.

### Key assumptions
- Vessel arrival rates are uncertain and follow probability distributions
- Service times vary based on vessel size and cargo volume
- Construction costs are linear per meter of quay length
- Waiting costs are linear per hour of vessel delay
- Solution must be feasible across all scenarios within uncertainty set

### Approach (step-by-step)
1. Define the robust optimization mathematical model with sets, parameters, and decision variables
2. Implement the uncertainty set using budgeted uncertainty approach
3. Create a concrete example with 3 berth segments and 2 vessel types
4. Solve the robust optimization problem using mixed-integer programming
5. Analyze results and perform sensitivity analysis

### What to look for in the results
- Optimal total quay length that balances construction and waiting costs
- Optimal berth configuration (number and sizes of berths)
- Worst-case expected waiting time under uncertainty
- Total cost breakdown (construction + expected waiting costs)

### Concrete example (from the source)
3 potential berth segments and 2 vessel types:
- Small vessels: L₁ = 200m, arrival rate λ₁ ∈ [8, 12] per week
- Large vessels: L₂ = 350m, arrival rate λ₂ ∈ [3, 7] per week  
- Service rates: μ₁ = 4 vessels/week, μ₂ = 2 vessels/week
- Costs: c_q = $25,000 per meter, c_w = $8,000 per hour
- Uncertainty budget: Γ = 1

In [1]:
from typing import Tuple, List, Dict, Set

# Import required packages for robust optimization
import pulp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

# Set up visualization style
plt.style.use('default')
sns.set_palette("husl")

In [2]:
# Define data structures for berth design problem
from dataclasses import dataclass
from typing import List, Dict, Tuple

@dataclass
class VesselType:
    """Represents a vessel type with its characteristics"""
    id: str
    length: float  # meters
    min_arrival_rate: float  # vessels per week
    max_arrival_rate: float  # vessels per week
    service_rate: float  # vessels per week
    
@dataclass
class BerthSegment:
    """Represents a potential berth segment"""
    id: str
    min_length: float  # meters
    max_length: float  # meters

@dataclass
class CostParameters:
    """Cost parameters for the optimization"""
    construction_cost_per_meter: float  # dollars
    waiting_cost_per_hour: float  # dollars
    
@dataclass
class UncertaintyParameters:
    """Uncertainty parameters for robust optimization"""
    budget: float  # uncertainty budget Gamma
    scenarios: List[Dict]  # list of scenarios

In [ ]:
# Create the concrete example from the source
def create_concrete_example():
    """Create the concrete example with 3 berth segments and 2 vessel types"""
    
    # Define vessel types
    vessel_types = [
        VesselType("Small", 200, 8, 12, 4),  # L₁ = 200m, λ₁ ∈ [8, 12], μ₁ = 4
        VesselType("Large", 350, 3, 7, 2)   # L₂ = 350m, λ₂ ∈ [3, 7], μ₂ = 2
    ]
    
    # Define berth segments
    berth_segments = [
        BerthSegment("B1", 200, 450),  # Min 200m, Max 450m
        BerthSegment("B2", 200, 450),  # Min 200m, Max 450m
        BerthSegment("B3", 200, 450)   # Min 200m, Max 450m
    ]
    
    # Define cost parameters
    costs = CostParameters(
        construction_cost_per_meter = 25000,  # $25,000 per meter
        waiting_cost_per_hour = 8000          # $8,000 per hour
    )
    
    # Define uncertainty parameters
    uncertainty = UncertaintyParameters(
        budget = 1.0,  # Γ = 1
        scenarios = [
            {"Small": 8, "Large": 3},   # Low demand scenario
            {"Small": 10, "Large": 5},  # Medium demand scenario  
            {"Small": 12, "Large": 7}   # High demand scenario
        ]
    )
    
    return vessel_types, berth_segments, costs, uncertainty

# Create the example
vessels, berths, costs, uncertainty = create_concrete_example()

print("Vessel Types:")
for v in vessels:
    print(f"  {v.id}: Length={v.length}m, Arrival Rate=[{v.min_arrival_rate}, {v.max_arrival_rate}]/week, Service Rate={v.service_rate}/week")

print("\nBerth Segments:")
for b in berths:
    print(f"  {b.id}: Length Range=[{b.min_length}, {b.max_length}]m")

print("\nCost Parameters:")
print(f"  Construction Cost: ${costs.construction_cost_per_meter:,} per meter")
print(f"  Waiting Cost: ${costs.waiting_cost_per_hour:,} per hour")

print("\nUncertainty Parameters:")
print(f"  Budget (Γ): {uncertainty.budget}")
print("  Scenarios:")
for i, scenario in enumerate(uncertainty.scenarios):
    print(f"    Scenario {i+1}: {scenario}")

Vessel Types:
  Small: Length=200m, Arrival Rate=[8, 12]/week, Service Rate=4/week
  Large: Length=350m, Arrival Rate=[3, 7]/week, Service Rate=2/week

Berth Segments:
  B1: Length Range=[200, 450]m
  B2: Length Range=[200, 450]m
  B3: Length Range=[200, 450]m

Cost Parameters:
  Construction Cost: $25,000 per meter
  Waiting Cost: $8,000 per hour

Uncertainty Parameters:
  Budget (Γ): 1.0
  Scenarios:
    Scenario 1: {'Small': 8, 'Large': 3}
    Scenario 2: {'Small': 10, 'Large': 5}
    Scenario 3: {'Small': 12, 'Large': 7}


In [ ]:
# Calculate waiting time using queuing theory formulas
def calculate_waiting_time(arrival_rates: Dict[str, float], 
                         service_rates: Dict[str, float],
                         num_berths: int) -> float:
    """Calculate expected waiting time using M/G/c queuing approximation"""
    
    # Total arrival rate and service rate
    total_arrival_rate = sum(arrival_rates.values())
    weighted_service_rate = sum(arrival_rates[v] * service_rates[v] 
                                for v in arrival_rates.keys()) / total_arrival_rate
    
    # System utilization
    utilization = total_arrival_rate / (num_berths * weighted_service_rate)
    
    # Prevent numerical issues
    if utilization >= 1.0:
        return 999.0  # Very large waiting time for overloaded system
    
    # Pollaczek-Khinchine formula approximation for M/G/c queue
    # Using simplified formula for demonstration
    if num_berths == 1:
        # M/G/1 queue formula
        waiting_time = (utilization**2) / (2 * (1 - utilization)) * (1 / weighted_service_rate)
    else:
        # M/G/c approximation (extended Pollaczek-Khinchine)
        # This is a simplified approximation for multi-server queues
        erlang_c = ( (num_berths * utilization)**num_berths / np.math.factorial(num_berths) ) / \
                   ( sum( (num_berths * utilization)**k / np.math.factorial(k) for k in range(num_berths) ) + \
                    ( (num_berths * utilization)**num_berths / np.math.factorial(num_berths) ) * (1 / (1 - utilization)) )
        
        waiting_time = (erlang_c / (num_berths * weighted_service_rate * (1 - utilization))) * \
                      (1 + (utilization**2) / (2 * (1 - utilization)))
    
    return waiting_time * 168  # Convert from weeks to hours (assuming 168 hours per week)

# Test the waiting time calculation
test_rates = {"Small": 10, "Large": 5}
test_services = {"Small": 4, "Large": 2}

for num_berths in range(1, 4):
    wait_time = calculate_waiting_time(test_rates, test_services, num_berths)
    print(f"Berths: {num_berths}, Waiting Time: {wait_time:.2f} hours")

Berths: 1, Waiting Time: 999.00 hours
Berths: 2, Waiting Time: 999.00 hours
Berths: 3, Waiting Time: 999.00 hours


In [ ]:
# Implement the robust optimization model
def solve_robust_berth_design(vessel_types: List[VesselType],
                             berth_segments: List[BerthSegment],
                             costs: CostParameters,
                             uncertainty: UncertaintyParameters) -> Dict:
    """Solve robust berth design problem using mixed-integer programming"""
    
    # Create the optimization problem
    prob = pulp.LpProblem("Robust_Berth_Design", pulp.LpMinimize)
    
    # Decision variables
    # x[b] = 1 if berth segment b is constructed
    x = {}
    for b, segment in enumerate(berth_segments):
        x[b] = pulp.LpVariable(f"x_{b}", cat='Binary')
    
    # l[b] = length of berth segment b
    l = {}
    for b, segment in enumerate(berth_segments):
        l[b] = pulp.LpVariable(f"l_{b}", lowBound=0, cat='Continuous')
    
    # W[s] = expected waiting time in scenario s
    W = {}
    for s, scenario in enumerate(uncertainty.scenarios):
        W[s] = pulp.LpVariable(f"W_{s}", lowBound=0, cat='Continuous')
    
    # Auxiliary variables for linearization (to handle max in objective)
    W_max = pulp.LpVariable("W_max", lowBound=0, cat='Continuous')
    
    # Objective function: minimize construction cost + worst-case waiting cost
    construction_cost = costs.construction_cost_per_meter * pulp.lpSum(l[b] for b in range(len(berth_segments)))
    waiting_cost = costs.waiting_cost_per_hour * W_max
    
    prob += construction_cost + waiting_cost, "Total_Cost"
    
    # Constraint 1: Total quay length definition
    prob += pulp.lpSum(l[b] for b in range(len(berth_segments))) == pulp.lpSum(l[b] for b in range(len(berth_segments))), "Total_Length_Definition"
    
    # Constraint 2: Berth length bounds
    for b, segment in enumerate(berth_segments):
        prob += l[b] >= segment.min_length * x[b], f"Min_Length_{b}"
        prob += l[b] <= segment.max_length * x[b], f"Max_Length_{b}"
    
    # Constraint 3: W_max definition (W_max >= W[s] for all scenarios)
    for s, scenario in enumerate(uncertainty.scenarios):
        prob += W_max >= W[s], f"W_Max_Def_{s}"
    
    # Constraint 4: Waiting time calculation for each scenario
    for s, scenario in enumerate(uncertainty.scenarios):
        # Get arrival rates for this scenario
        arrival_rates = {}
        service_rates = {}
        
        for vessel in vessel_types:
            arrival_rates[vessel.id] = scenario[vessel.id]
            service_rates[vessel.id] = vessel.service_rate
        
        # Number of operating berths
        num_berths = pulp.lpSum(x[b] for b in range(len(berth_segments)))
        
        # Calculate utilization
        total_arrival = sum(arrival_rates.values())
        weighted_service = sum(arrival_rates[v] * service_rates[v] for v in arrival_rates.keys()) / total_arrival
        utilization = total_arrival / (num_berths * weighted_service)
        
        # Simplified waiting time approximation (linearized)
        # For robust optimization, we use a conservative approximation
        # W[s] >= base_waiting_time * utilization_factor
        
        # Base waiting time parameters (conservative estimates)
        base_wait_small = 2.0  # hours for small vessels
        base_wait_large = 4.0  # hours for large vessels
        
        # Conservative waiting time estimate
        total_vessels = sum(arrival_rates.values())
        estimated_wait = (base_wait_small * arrival_rates["Small"] + base_wait_large * arrival_rates["Large"]) / total_vessels
        
        # Scale by utilization (conservative factor)
        utilization_factor = 1.0 / (1.0 - utilization + 0.01)  # Add small constant to avoid division by zero
        
        prob += W[s] >= estimated_wait * utilization_factor, f"Waiting_Time_{s}"
    
    # Solve the problem
    print("Solving robust optimization problem...")
    prob.solve(pulp.PULP_CBC_CMD(msg=False))
    
    # Extract results
    status = pulp.LpStatus[prob.status]
    objective_value = pulp.value(prob.objective)
    
    # Extract berth decisions
    constructed_berths = []
    berth_lengths = {}
    total_quay_length = 0
    
    for b, segment in enumerate(berth_segments):
        if pulp.value(x[b]) > 0.5:
            constructed_berths.append(segment.id)
            berth_lengths[segment.id] = pulp.value(l[b])
            total_quay_length += pulp.value(l[b])
    
    # Extract waiting times
    waiting_times = {}
    for s, scenario in enumerate(uncertainty.scenarios):
        waiting_times[f"Scenario_{s+1}"] = pulp.value(W[s])
    
    worst_case_wait = pulp.value(W_max)
    
    return {
        'status': status,
        'objective_value': objective_value,
        'constructed_berths': constructed_berths,
        'berth_lengths': berth_lengths,
        'total_quay_length': total_quay_length,
        'waiting_times': waiting_times,
        'worst_case_wait': worst_case_wait,
        'construction_cost': construction_cost.value(),
        'waiting_cost': waiting_cost.value(),
        'problem': prob
    }

In [ ]:
try:
    # Solve the concrete example
    result = solve_robust_berth_design(vessels, berths, costs, uncertainty)
    
    print(f"Optimization Status: {result['status']}")
    print(f"Total Cost: ${result['objective_value']:,.2f}")
    print(f"Construction Cost: ${result['construction_cost']:,.2f}")
    print(f"Worst-case Waiting Cost: ${result['waiting_cost']:,.2f}")
    print(f"Total Quay Length: {result['total_quay_length']:.1f} meters")
    print(f"Worst-case Waiting Time: {result['worst_case_wait']:.2f} hours")
    
    print("\nConstructed Berths:")
    for berth_id in result['constructed_berths']:
        length = result['berth_lengths'][berth_id]
        print(f"  {berth_id}: {length:.1f} meters")
    
    print("\nScenario Waiting Times:")
    for scenario, wait_time in result['waiting_times'].items():
        print(f"  {scenario}: {wait_time:.2f} hours")
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: unsupported operand type(s) for /: 'int' and 'LpAffineExpression'...]

In [ ]:
try:
    # Verify the solution manually with detailed calculations
    def verify_solution(result: Dict, vessels: List[VesselType], 
                       uncertainty: UncertaintyParameters, costs: CostParameters):
        """Verify the solution with detailed queuing calculations"""
        
        print("Solution Verification:")
        print("=" * 60)
        
        num_berths = len(result['constructed_berths'])
        total_length = result['total_quay_length']
        
        print(f"Number of Berths: {num_berths}")
        print(f"Total Quay Length: {total_length:.1f} meters")
        print(f"Average Berth Length: {total_length/num_berths:.1f} meters")
        
        # Check each scenario with detailed queuing analysis
        service_rates = {v.id: v.service_rate for v in vessels}
        
        for s, scenario in enumerate(uncertainty.scenarios):
            print(f"\nScenario {s+1} Analysis:")
            print(f"  Arrival Rates: {scenario}")
            
            # Calculate detailed waiting time
            actual_wait = calculate_waiting_time(scenario, service_rates, num_berths)
            model_wait = result['waiting_times'][f"Scenario_{s+1}"]
            
            print(f"  Calculated Waiting Time: {actual_wait:.2f} hours")
            print(f"  Model Waiting Time: {model_wait:.2f} hours")
            print(f"  Difference: {abs(actual_wait - model_wait):.2f} hours")
            
            # Calculate utilization
            total_arrival = sum(scenario.values())
            weighted_service = sum(scenario[v] * service_rates[v] for v in scenario.keys()) / total_arrival
            utilization = total_arrival / (num_berths * weighted_service)
            
            print(f"  System Utilization: {utilization:.3f} ({utilization*100:.1f}%)")
            print(f"  Weekly Capacity: {num_berths * weighted_service:.1f} vessels")
            print(f"  Weekly Demand: {total_arrival:.1f} vessels")
    
    # Verify the solution
    verify_solution(result, vessels, uncertainty, costs)
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: name 'result' is not defined...]

In [ ]:
try:
    # Visualize the berth design solution
    def visualize_solution(result: Dict, vessels: List[VesselType]):
        """Create comprehensive visualizations of the berth design solution"""
        
        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
        
        # Plot 1: Berth configuration visualization
        ax1.set_title('Optimal Berth Configuration', fontsize=14, fontweight='bold')
        ax1.set_xlabel('Berth Segment')
        ax1.set_ylabel('Length (meters)')
        ax1.grid(True, alpha=0.3)
        
        berth_ids = list(result['berth_lengths'].keys())
        berth_lengths = list(result['berth_lengths'].values())
        
        colors = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D']
        bars = ax1.bar(berth_ids, berth_lengths, color=colors[:len(berth_ids)], alpha=0.8)
        
        # Add value labels on bars
        for bar, length in zip(bars, berth_lengths):
            height = bar.get_height()
            ax1.text(bar.get_x() + bar.get_width()/2., height + 5,
                    f'{length:.0f}m', ha='center', va='bottom', fontweight='bold')
        
        # Plot 2: Cost breakdown
        ax2.set_title('Cost Breakdown Analysis', fontsize=14, fontweight='bold')
        ax2.set_ylabel('Cost ($ millions)')
        ax2.grid(True, alpha=0.3, axis='y')
        
        cost_categories = ['Construction', 'Waiting']
        cost_values = [result['construction_cost']/1e6, result['waiting_cost']/1e6]
        
        bars = ax2.bar(cost_categories, cost_values, color=['#2E86AB', '#A23B72'], alpha=0.8)
        
        for bar, value in zip(bars, cost_values):
            height = bar.get_height()
            ax2.text(bar.get_x() + bar.get_width()/2., height + 0.1,
                    f'${value:.2f}M', ha='center', va='bottom', fontweight='bold')
        
        # Plot 3: Scenario waiting times
        ax3.set_title('Waiting Times by Scenario', fontsize=14, fontweight='bold')
        ax3.set_xlabel('Scenario')
        ax3.set_ylabel('Waiting Time (hours)')
        ax3.grid(True, alpha=0.3)
        
        scenarios = list(result['waiting_times'].keys())
        wait_times = list(result['waiting_times'].values())
        
        bars = ax3.bar(scenarios, wait_times, color=['#F18F01', '#C73E1D', '#2E86AB'], alpha=0.8)
        
        # Add worst-case line
        ax3.axhline(y=result['worst_case_wait'], color='red', linestyle='--', 
                    label=f'Worst Case: {result["worst_case_wait"]:.2f}h', alpha=0.7)
        ax3.legend()
        
        for bar, wait_time in zip(bars, wait_times):
            height = bar.get_height()
            ax3.text(bar.get_x() + bar.get_width()/2., height + 0.1,
                    f'{wait_time:.1f}h', ha='center', va='bottom', fontweight='bold')
        
        # Plot 4: Vessel type compatibility
        ax4.set_title('Vessel Type Compatibility', fontsize=14, fontweight='bold')
        ax4.set_xlabel('Vessel Type')
        ax4.set_ylabel('Length (meters)')
        ax4.grid(True, alpha=0.3)
        
        vessel_names = [v.id for v in vessels]
        vessel_lengths = [v.length for v in vessels]
        
        # Show vessel lengths vs average berth length
        avg_berth_length = result['total_quay_length'] / len(result['constructed_berths'])
        
        x_pos = np.arange(len(vessel_names))
        bars = ax4.bar(x_pos, vessel_lengths, color=['#2E86AB', '#A23B72'], alpha=0.8, width=0.4)
        
        # Add average berth length line
        ax4.axhline(y=avg_berth_length, color='green', linestyle='--', 
                    label=f'Avg Berth: {avg_berth_length:.0f}m', alpha=0.7)
        ax4.legend()
        
        ax4.set_xticks(x_pos)
        ax4.set_xticklabels(vessel_names)
        
        for bar, length in zip(bars, vessel_lengths):
            height = bar.get_height()
            ax4.text(bar.get_x() + bar.get_width()/2., height + 5,
                    f'{length}m', ha='center', va='bottom', fontweight='bold')
        
        plt.suptitle('Robust Berth Design Optimization Results', fontsize=16, fontweight='bold')
        plt.tight_layout()
        plt.show()
    
    # Visualize the solution
    visualize_solution(result, vessels)
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: name 'result' is not defined...]

In [ ]:
try:
    # Sensitivity analysis on uncertainty budget
    def sensitivity_analysis():
        """Perform sensitivity analysis on uncertainty budget parameter"""
        
        # Test different uncertainty budget values
        gamma_values = [0.5, 1.0, 1.5, 2.0]
        results = []
        
        for gamma in gamma_values:
            # Modify uncertainty parameters
            modified_uncertainty = UncertaintyParameters(
                budget=gamma,
                scenarios=uncertainty.scenarios
            )
            
            # Solve with modified uncertainty budget
            result_modified = solve_robust_berth_design(vessels, berths, costs, modified_uncertainty)
            
            results.append({
                'gamma': gamma,
                'total_cost': result_modified['objective_value'],
                'construction_cost': result_modified['construction_cost'],
                'waiting_cost': result_modified['waiting_cost'],
                'quay_length': result_modified['total_quay_length'],
                'worst_wait': result_modified['worst_case_wait'],
                'num_berths': len(result_modified['constructed_berths'])
            })
        
        # Create sensitivity analysis visualization
        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
        
        # Plot 1: Total cost vs uncertainty budget
        gammas = [r['gamma'] for r in results]
        total_costs = [r['total_cost']/1e6 for r in results]
        
        ax1.plot(gammas, total_costs, 'bo-', linewidth=2, markersize=8)
        ax1.set_xlabel('Uncertainty Budget (Γ)')
        ax1.set_ylabel('Total Cost ($ millions)')
        ax1.set_title('Impact of Uncertainty Budget on Total Cost')
        ax1.grid(True, alpha=0.3)
        ax1.set_xticks(gammas)
        
        # Plot 2: Cost components vs uncertainty budget
        construction_costs = [r['construction_cost']/1e6 for r in results]
        waiting_costs = [r['waiting_cost']/1e6 for r in results]
        
        ax2.plot(gammas, construction_costs, 'g^-', linewidth=2, markersize=8, label='Construction')
        ax2.plot(gammas, waiting_costs, 'r^-', linewidth=2, markersize=8, label='Waiting')
        ax2.set_xlabel('Uncertainty Budget (Γ)')
        ax2.set_ylabel('Cost ($ millions)')
        ax2.set_title('Cost Components vs Uncertainty Budget')
        ax2.grid(True, alpha=0.3)
        ax2.legend()
        ax2.set_xticks(gammas)
        
        # Plot 3: Quay length vs uncertainty budget
        quay_lengths = [r['quay_length'] for r in results]
        
        ax3.plot(gammas, quay_lengths, 'mo-', linewidth=2, markersize=8)
        ax3.set_xlabel('Uncertainty Budget (Γ)')
        ax3.set_ylabel('Total Quay Length (meters)')
        ax3.set_title('Quay Length vs Uncertainty Budget')
        ax3.grid(True, alpha=0.3)
        ax3.set_xticks(gammas)
        
        # Plot 4: Number of berths vs uncertainty budget
        num_berths = [r['num_berths'] for r in results]
        
        ax4.bar(range(len(gamma_values)), num_berths, color='orange', alpha=0.7)
        ax4.set_xlabel('Uncertainty Budget (Γ)')
        ax4.set_ylabel('Number of Berths')
        ax4.set_title('Number of Berths vs Uncertainty Budget')
        ax4.set_xticks(range(len(gamma_values)))
        ax4.set_xticklabels([f'{g}' for g in gamma_values])
        ax4.grid(True, alpha=0.3, axis='y')
        
        # Add value labels on bars
        for i, count in enumerate(num_berths):
            ax4.text(i, count + 0.05, str(count), ha='center', va='bottom', fontweight='bold')
        
        plt.suptitle('Sensitivity Analysis: Impact of Uncertainty Budget', fontsize=16, fontweight='bold')
        plt.tight_layout()
        plt.show()
        
        # Print detailed results
        print("Sensitivity Analysis Results:")
        print("=" * 80)
        for r in results:
            print(f"Uncertainty Budget (Γ): {r['gamma']}")
            print(f"  Total Cost: ${r['total_cost']:,.2f}")
            print(f"  Construction Cost: ${r['construction_cost']:,.2f}")
            print(f"  Waiting Cost: ${r['waiting_cost']:,.2f}")
            print(f"  Quay Length: {r['quay_length']:.1f} meters")
            print(f"  Number of Berths: {r['num_berths']}")
            print(f"  Worst-case Wait: {r['worst_wait']:.2f} hours")
            print()
    
    # Run sensitivity analysis
    sensitivity_analysis()
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: unsupported operand type(s) for /: 'int' and 'LpAffineExpression'...]

### Why this Tier exists vs earlier Tiers
This is Tier 1, the foundational mathematical approach. It provides:
- **Robust optimal solutions** that protect against uncertainty
- **Mathematical rigor** with provable feasibility guarantees
- **Baseline for comparison** with heuristic and metaheuristic methods
- **Understanding of problem structure** through mathematical modeling

### Pros / Cons vs this Tier
**Advantages:**
- Guarantees robust optimal solution under uncertainty
- Handles all constraints exactly with mathematical precision
- Provides optimality gap information and feasibility guarantees
- Reproducible and deterministic results
- Explicit consideration of uncertainty through robust optimization

**Disadvantages:**
- Computationally expensive for large instances
- Requires specialized optimization software and expertise
- May not scale to real-world problem sizes with many scenarios
- Limited flexibility for dynamic real-time decisions
- Requires accurate uncertainty set specification

### When to use this Tier
- **Strategic infrastructure planning** where robustness is critical
- **Capital investment decisions** requiring uncertainty protection
- **Regulatory compliance** requiring provable constraint satisfaction
- **Benchmark development** for comparing other methods
- **Academic and research settings** where mathematical rigor is required
- **Safety-critical applications** where optimal solutions are mandatory